In [9]:
from godot_rl.core.godot_env import GodotEnv
from dataclasses import dataclass
from collections import deque

from torch.distributions.categorical import Categorical

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Create GodotEnv
env = GodotEnv(show_window=True)

# Get action space info
# GodotEnv returns list of action dicts per agent
# Each agent has: accelerate_forward (3), accelerate_sideways (3), turn (3), shoot (2)
print(f"Environment: GodotEnv")
print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")

Using device: mps
No game binary has been provided, please press PLAY in the Godot editor
waiting for remote GODOT connection on port 11008
connection established
action space [{'accelerate_forward': {'size': 3, 'action_type': 'discrete'}, 'accelerate_sideways': {'size': 3, 'action_type': 'discrete'}, 'turn': {'size': 3, 'action_type': 'discrete'}, 'shoot': {'size': 2, 'action_type': 'discrete'}}]
observation space [{'right_eye': {'size': [3, 300, 320], 'space': 'box'}, 'left_eye': {'size': [3, 300, 320], 'space': 'box'}, 'hp': {'size': [1], 'space': 'box'}}]
Environment: GodotEnv
Action space: Tuple(Discrete(3), Discrete(3), Discrete(2), Discrete(3))
Observation space: Dict('hp': Box(-1.0, 1.0, (1,), float32), 'left_eye': Box(-1.0, 1.0, (3, 300, 320), float32), 'right_eye': Box(-1.0, 1.0, (3, 300, 320), float32))


In [10]:
# Test observation structure
obs, _ = env.reset()
print(f"Number of agents: {len(obs)}")
print(f"Agent 0 observation keys: {obs[0].keys()}")

# Decode and check image shapes
right_eye_hex = obs[0]['right_eye']
right_eye_bytes = bytes.fromhex(right_eye_hex)
right_eye_arr = np.frombuffer(right_eye_bytes, dtype=np.uint8)
print(f"Right eye raw data length: {len(right_eye_arr)}")

left_eye_hex = obs[0]['left_eye']
left_eye_bytes = bytes.fromhex(left_eye_hex)
left_eye_arr = np.frombuffer(left_eye_bytes, dtype=np.uint8)
print(f"Left eye raw data length: {len(left_eye_arr)}")

hp = obs[0]['hp']
print(f"HP: {hp}")

Number of agents: 1
Agent 0 observation keys: dict_keys(['right_eye', 'left_eye', 'hp'])
Right eye raw data length: 288000
Left eye raw data length: 288000
HP: [6]


In [11]:
# GodotEnv Preprocessor
class GodotPreprocessor:
    """Preprocesses GodotEnv observations.
    
    Combines:
    - left_eye (320 width, 300 height) and right_eye (320 width, 300 height) images
    - Previous output (out_next) from model
    - Current HP
    
    Process:
    - Stack left_eye and right_eye vertically: (3, 600, 320)
    - HP embedded as (3, 40, 320)
    - Combined observation: (3, 640, 320)
    - Concatenate with out_next (3, 640, 320) horizontally: (3, 640, 640)
    
    Final input shape: (3, 640, 640)
    """
    
    def __init__(self):
        self.original_eye_size = (300, 320)  # (height, width) from GodotEnv
        self.hp_height = 40  # Height for HP bar
        self.out_next_shape = (3, 640, 320)  # Shape of model output / previous output
        
    def reset(self):
        """No state to clear."""
        pass
    
    def decode_eye(self, eye_hex):
        """Decode hex string to numpy array (H, W, 3)."""
        eye_bytes = bytes.fromhex(eye_hex)
        arr = np.frombuffer(eye_bytes, dtype=np.uint8)
        # Original shape is (height=300, width=320, 3)
        img = arr.reshape(self.original_eye_size[0], self.original_eye_size[1], 3)
        return img
    
    def get_state(self, obs_dict):
        """Process observation dict from a single agent.
        
        Args:
            obs_dict: dict with 'left_eye', 'right_eye', 'hp' keys
            
        Returns:
            combined_eyes: (3, 600, 320) - vertically stacked eyes
            hp: float - health value normalized to [0, 1]
        """
        # Decode eyes - each is (300, 320, 3) = (height, width, channels)
        left_eye = self.decode_eye(obs_dict['left_eye'])  # (300, 320, 3)
        right_eye = self.decode_eye(obs_dict['right_eye'])  # (300, 320, 3)
        
        # Stack vertically without resizing: (600, 320, 3)
        combined = np.concatenate([left_eye, right_eye], axis=0)
        
        # Normalize to [0, 1] and transpose to (3, 600, 320)
        combined = combined.astype(np.float32) / 255.0
        combined = np.transpose(combined, (2, 0, 1))
        
        # Get HP (it's a list with one element, normalize assuming max HP is 100)
        hp_value = obs_dict['hp']
        if isinstance(hp_value, (list, np.ndarray)):
            hp_value = hp_value[0]  # Extract first element from list
        hp = float(hp_value) / 100.0  # Normalize HP
        
        return combined, hp
    
    def combine_state_with_out_next_and_hp(self, combined_eyes, hp, out_next=None):
        """Combine eyes, hp, and out_next into final input.
        
        Args:
            combined_eyes: (3, 600, 320) - vertically stacked eyes
            hp: float - normalized health value
            out_next: (3, 640, 320) or None - previous model output
            
        Returns:
            combined: (3, 640, 640) - full input for model
        """
        _, eye_h, eye_w = combined_eyes.shape  # (3, 600, 320)
        
        # Create HP embedding: broadcast hp value to (3, 40, 320)
        hp_embed = np.full((3, self.hp_height, eye_w), hp, dtype=np.float32)
        
        # Concatenate vertically: eyes (3, 600, 320) + hp (3, 40, 320) = (3, 640, 320)
        current_obs = np.concatenate([combined_eyes, hp_embed], axis=1)
        
        # If out_next is None, create zeros (3, 640, 320)
        if out_next is None:
            out_next = np.zeros(self.out_next_shape, dtype=np.float32)
        
        # Concatenate horizontally: current_obs (3, 640, 320) + out_next (3, 640, 320) = (3, 640, 640)
        combined = np.concatenate([current_obs, out_next], axis=2)
        
        return combined
    
    def get_input_shape(self):
        """Return the expected input shape for the model."""
        return (3, 640, 640)
    
    def get_out_next_shape(self):
        """Return the expected shape of out_next (model output)."""
        return self.out_next_shape  # (3, 640, 320)


# Test the preprocessor
preprocessor = GodotPreprocessor()
combined_eyes, hp = preprocessor.get_state(obs[0])
print(f"Combined eyes shape: {combined_eyes.shape}")  # Should be (3, 600, 320)
print(f"HP value: {hp}")

combined_full = preprocessor.combine_state_with_out_next_and_hp(combined_eyes, hp)
print(f"Full combined shape (with zeros out_next): {combined_full.shape}")  # Should be (3, 640, 640)
print(f"Expected input shape: {preprocessor.get_input_shape()}")  # (3, 640, 640)
print(f"Expected out_next shape: {preprocessor.get_out_next_shape()}")  # (3, 640, 320)

Combined eyes shape: (3, 600, 320)
HP value: 0.06
Full combined shape (with zeros out_next): (3, 640, 640)
Expected input shape: (3, 640, 640)
Expected out_next shape: (3, 640, 320)


In [12]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

# Action dimensions for GodotEnv
ACTION_DIMS = {
    'accelerate_forward': 3,
    'accelerate_sideways': 3,
    'turn': 3,
    'shoot': 2
}
TOTAL_ACTIONS = sum(ACTION_DIMS.values())  # 3 + 3 + 3 + 2 = 11


class MobileNetV3PolicyGodot(nn.Module):
    """MobileNetV3-based policy for GodotEnv with multi-discrete actions.
    
    Input: (3, 640, 640) - combined (eyes + hp) + out_next
    Output: (3, 640, 320) - used for action selection and as out_next for next step
    
    Action space: Multi-discrete
    - accelerate_forward: 3 choices
    - accelerate_sideways: 3 choices  
    - turn: 3 choices
    - shoot: 2 choices
    """
    
    def __init__(self, pretrained=True, freeze_backbone=False):
        super(MobileNetV3PolicyGodot, self).__init__()
        self.action_dims = ACTION_DIMS
        self.total_actions = TOTAL_ACTIONS
        self.out_h = 640  # Output height
        self.out_w = 320  # Output width
        
        # Load pretrained MobileNetV3-Small
        if pretrained:
            weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
            backbone = mobilenet_v3_small(weights=weights)
        else:
            backbone = mobilenet_v3_small(weights=None)
        
        # Keep only the feature extractor
        # Input: (batch, 3, 640, 640)
        # MobileNetV3-Small features output: (batch, 576, H/32, W/32)
        # For 640x640 input: output is (batch, 576, 20, 20)
        self.features = backbone.features
        
        if freeze_backbone:
            for param in self.features.parameters():
                param.requires_grad = False
        
        # Upsample to output shape (3, 640, 320)
        self.upsample_conv1 = nn.Sequential(
            nn.Conv2d(576, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.upsample_conv2 = nn.Sequential(
            nn.Conv2d(256, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.upsample_conv3 = nn.Conv2d(64, 3, kernel_size=3, padding=1)
        
        self._init_upsample_weights()
    
    def _init_upsample_weights(self):
        for m in [self.upsample_conv1, self.upsample_conv2, self.upsample_conv3]:
            for layer in m.modules() if isinstance(m, nn.Sequential) else [m]:
                if isinstance(layer, nn.Conv2d):
                    nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        """Forward pass, returns (batch, 3, 640, 320)."""
        # Extract features: (batch, 3, 640, 640) -> (batch, 576, 20, 20)
        features = self.features(x)
        
        # Upsample using interpolate for flexible sizing
        x = self.upsample_conv1(features)  # (batch, 256, 20, 20)
        x = F.interpolate(x, size=(160, 80), mode='bilinear', align_corners=False)
        
        x = self.upsample_conv2(x)  # (batch, 64, 160, 80)
        x = F.interpolate(x, size=(self.out_h, self.out_w), mode='bilinear', align_corners=False)
        
        # Final conv: (batch, 64, 640, 320) -> (batch, 3, 640, 320)
        out = self.upsample_conv3(x)
        return out
    
    def extract_action_logits_from_crops(self, out):
        """Extract logits for each action choice from 11 consecutive 3x8x8 crops.
        
        Args:
            out: tensor (batch, 3, 640, 320) - model output
            
        Returns:
            logits: list of tensors, one per action type with shape (batch, action_size)
        """
        # Extract 11 crops: each crop is 3x2x2
        crop_values = []
        for i in range(self.total_actions):
            crop = out[:, :, 16:17, (16 + i*2):(16 + (i+1)*2)]  # (batch, 3, 2, 2)
            crop_mean = crop.mean(dim=(1, 2, 3))  # (batch,) - mean across channels and spatial dims
            crop_values.append(crop_mean)
        
        # Stack all crop values: (batch, 11)
        all_logits = torch.stack(crop_values, dim=1)  # (batch, 11)
        
        # Split into action-specific logits
        logits = []
        idx = 0
        for action_name, action_size in self.action_dims.items():
            action_logits = all_logits[:, idx:idx + action_size]  # (batch, action_size)
            logits.append(action_logits)
            idx += action_size
        
        return logits
    
    def act(self, combined_state):
        """Select actions and generate out_next for next step.
        
        Args:
            combined_state: (3, 640, 640) - combined (eyes + hp) + out_next
            
        Returns:
            actions: dict with action keys
            log_prob: tensor - sum of log probabilities
            out_next: numpy array (3, 640, 320) - output for next step's input
        """
        if len(combined_state.shape) == 3:
            combined_state = combined_state[np.newaxis, ...]
        
        state_tensor = torch.FloatTensor(combined_state).to(device)
        out = self.forward(state_tensor)  # (1, 3, 640, 320)
        
        # Extract action logits from first 11 3x8x8 crops
        action_logits_list = self.extract_action_logits_from_crops(out)
        
        # Select actions for each action dimension
        actions = {}
        log_probs = []
        
        for (action_name, action_size), action_logits in zip(self.action_dims.items(), action_logits_list):
            probs = F.softmax(action_logits, dim=1).cpu()
            
            m = Categorical(probs)
            action = m.sample()
            log_prob = m.log_prob(action)
            
            actions[action_name] = action.item()
            log_probs.append(log_prob)
        
        # Sum log probs for total action probability
        total_log_prob = torch.stack(log_probs).sum()
        
        # out_next is directly the model output (3, 640, 320)
        out_next = out.squeeze(0).detach().cpu().numpy()  # (3, 640, 320)
        
        return actions, total_log_prob, out_next


# Create and test the model
policy = MobileNetV3PolicyGodot(pretrained=True, freeze_backbone=False).to(device)
print(f"MobileNetV3 Policy for GodotEnv:")
print(policy)

# Count parameters
total_params = sum(p.numel() for p in policy.parameters())
trainable_params = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass with correct input size
test_input = torch.randn(1, 3, 640, 640).to(device)
test_output = policy(test_input)
print(f"\nTest input shape: {test_input.shape}")  # (1, 3, 640, 640)
print(f"Test output shape: {test_output.shape}")  # (1, 3, 640, 320)

MobileNetV3 Policy for GodotEnv:
MobileNetV3PolicyGodot(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
      

In [13]:
# Test the act function
combined_eyes, hp = preprocessor.get_state(obs[0])
combined_full = preprocessor.combine_state_with_out_next_and_hp(combined_eyes, hp)
print(f"Combined eyes shape: {combined_eyes.shape}")  # Should be (3, 600, 320)
print(f"Input shape: {combined_full.shape}")  # Should be (3, 640, 640)

actions, log_prob, out_next = policy.act(combined_full)
print(f"Actions: {actions}")
print(f"Log prob: {log_prob}")
print(f"Out next shape: {out_next.shape}")  # Should be (3, 640, 320)

# Test with out_next feedback
combined_full_with_out_next = preprocessor.combine_state_with_out_next_and_hp(combined_eyes, hp, out_next)
print(f"Input with out_next shape: {combined_full_with_out_next.shape}")  # Should be (3, 640, 640)

Combined eyes shape: (3, 600, 320)
Input shape: (3, 640, 640)
Actions: {'accelerate_forward': 2, 'accelerate_sideways': 0, 'turn': 0, 'shoot': 0}
Log prob: -3.9053103923797607
Out next shape: (3, 640, 320)
Input with out_next shape: (3, 640, 640)


In [14]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def extract_loss_and_lr_regions(out_next):
    """
    Extract the middle 3x8x8 region (LOSS region) and adjacent region (LR region) from model output.
    
    Model output shape: (3, 640, 320)
    LOSS region: center 3x8x8 - used for computing loss
    LR region: adjacent 3x8x8 to the right - used for determining learning rate
    
    Args:
        out_next: numpy array (3, 640, 320)
        
    Returns:
        loss_region: (3, 2, 2) - center region for loss
        lr_region: (3, 2, 2) - adjacent region for learning rate
    """
    # Model output is (3, 640, 320)
    # Center: height 640/2=320, width 320/2=160
    # LOSS region: [316:324, 156:164] (8x8 centered)
    h_center = 320
    w_center = 160
    half_size = 1
    
    loss_region = out_next[:, h_center-half_size:h_center+half_size, w_center-half_size:w_center+half_size]
    # LR region: adjacent to the right [316:324, 164:172]
    lr_region = out_next[:, h_center-half_size:h_center+half_size, w_center+half_size:w_center+half_size*2]
    
    return loss_region, lr_region


def compute_loss_from_region(loss_region):
    """
    Compute loss value from the LOSS region (3x8x8).
    Uses mean squared error from target (zero) plus variance to encourage diversity.
    
    Args:
        loss_region: torch tensor (3, 8, 8)
        
    Returns:
        loss: scalar tensor
    """
    # Use mean of absolute values as loss (encourage outputs to be non-zero but controlled)
    # Add variance term to encourage diverse outputs
    mean_abs = torch.abs(loss_region).mean()
    variance = loss_region.var()
    
    # Loss = mean absolute value - small variance bonus (encourage diversity)
    loss = mean_abs - 0.1 * variance
    return loss


def compute_lr_from_region(lr_region, lr_min=1e-6, lr_max=1e-4):
    """
    Compute learning rate from the LR region (3x2x2) normalized.
    
    Args:
        lr_region: numpy array (3, 2, 2)
        lr_min: minimum learning rate
        lr_max: maximum learning rate
        
    Returns:
        lr: float - learning rate in [lr_min, lr_max]
    """
    # Normalize the region to [0, 1] using sigmoid-like function
    lr_mean = np.mean(lr_region)
    # Use tanh to bound the value, then scale to [0, 1]
    normalized = (np.tanh(lr_mean) + 1) / 2  # Maps to [0, 1]
    
    # Scale to learning rate range
    lr = lr_min + normalized * (lr_max - lr_min)
    return float(lr)


def reinforce_godot(policy, optimizer, preprocessor, env, n_training_episodes, max_t, gamma, print_every, 
                   lr_min=1e-6, lr_max=1e-4, death_lr=1e-2, n_agents=8):
    """
    REINFORCE algorithm for GodotEnv with per-step updates.
    
    Key changes:
    1. Update parameters after every step (not mini-batches)
    2. Use middle 3x8x8 region from model output as loss
    3. Learning rate determined by adjacent 3x8x8 region
    4. When HP <= 0: compute negative loss gradient directly on current params with large LR
    
    Note: GodotEnv supports multiple agents. We train on agent 0 for simplicity.
    """
    scores_deque = deque(maxlen=100)
    scores = []
    best_score = -np.inf
    
    for i_episode in range(1, n_training_episodes + 1):
        # Reset environment and preprocessor
        obs, _ = env.reset()
        preprocessor.reset()

        # Get initial state for agent 0
        combined_eyes, hp = preprocessor.get_state(obs[0])
        
        # Initialize out_next as None (will be zeros on first step)
        out_next = None
        
        # Track total episode reward
        episode_reward = 0
        total_steps = 0
        survival_steps = 0  # Steps survived since last reset
        done = False
        
        for t in range(max_t):
            # Combine state with out_next and hp
            combined_state = preprocessor.combine_state_with_out_next_and_hp(
                combined_eyes, hp, out_next
            )
            
            # Forward pass to get output tensor (need gradients for loss)
            if len(combined_state.shape) == 3:
                state_batch = combined_state[np.newaxis, ...]
            else:
                state_batch = combined_state
            state_tensor = torch.FloatTensor(state_batch).to(device)
            
            # Get model output with gradients
            model_output = policy(state_tensor)  # (1, 3, 640, 320)
            
            # Extract LOSS and LR regions
            out_numpy = model_output.squeeze(0).detach().cpu().numpy()  # (3, 640, 320)
            loss_region_np, lr_region_np = extract_loss_and_lr_regions(out_numpy)
            
            # Compute dynamic learning rate from LR region
            dynamic_lr = compute_lr_from_region(lr_region_np, lr_min, lr_max)
            
            # Update optimizer learning rate
            for param_group in optimizer.param_groups:
                param_group['lr'] = dynamic_lr
            
            # Compute loss from LOSS region (need tensor version for backprop)
            loss_region_tensor = model_output[:, :, 316:324, 156:164]  # (1, 3, 8, 8)
            loss = compute_loss_from_region(loss_region_tensor.squeeze(0))
            
            # === Per-step parameter update ===
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
            optimizer.step()
            
            # Get actions from the output using 3x8x8 crops (using detached output)
            action_logits_list = policy.extract_action_logits_from_crops(model_output.detach())
            actions = {}
            for (action_name, action_size), action_logits in zip(policy.action_dims.items(), action_logits_list):
                probs = F.softmax(action_logits, dim=1).cpu()
                m = Categorical(probs)
                action = m.sample()
                actions[action_name] = action.item()
            
            # out_next for next step's input
            out_next = out_numpy
            
            # Format actions for all agents (use same actions for all)
            all_actions = []
            for _ in range(n_agents):
                agent_action = [
                    actions['accelerate_forward'],
                    actions['accelerate_sideways'],
                    actions['turn'],
                    actions['shoot']
                ]
                all_actions.append(agent_action)
            
            # Step environment
            obs, reward_list, terminated_list, truncated_list, _ = env.step(all_actions,True)
            # Get reward for agent 0
            reward = reward_list[0] if isinstance(reward_list, list) else reward_list
            done = (terminated_list[0] if isinstance(terminated_list, list) else terminated_list) or \
                   (truncated_list[0] if isinstance(truncated_list, list) else truncated_list)
            
            episode_reward += reward
            total_steps += 1
            survival_steps += 1
            
            # Get next state and HP
            combined_eyes, hp = preprocessor.get_state(obs[0])
            current_hp = hp * 100  # Denormalize HP for comparison
            
            # === Print status every 100 steps ===
            if total_steps % 100 == 0:
                print(f"  [Step {total_steps}] HP={current_hp:.1f}, LR={dynamic_lr:.2e}, Survived={survival_steps} steps")
            
            # === Handle death: apply negative loss gradient directly on current params ===
            if current_hp <= 0:
                print(f"  [Step {total_steps}] HP depleted! Applying negative loss update on current params...")
                
                # # === 展示死亡时机器人左右眼看到的图像 ===
                # left_eye_img = preprocessor.decode_eye(obs[0]['left_eye'])   # (300, 320, 3)
                # right_eye_img = preprocessor.decode_eye(obs[0]['right_eye']) # (300, 320, 3)
                
                # fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                # axes[0].imshow(left_eye_img)
                # axes[0].set_title(f'左眼 (Death @ Step {total_steps}, Survived {survival_steps} steps)')
                # axes[0].axis('off')
                # axes[1].imshow(right_eye_img)
                # axes[1].set_title(f'右眼 (Death @ Step {total_steps}, Survived {survival_steps} steps)')
                # axes[1].axis('off')
                # plt.suptitle(f'HP降为零时的视野 (Episode {i_episode}, Step {total_steps})', fontsize=14)
                # plt.tight_layout()
                # plt.show()
                
                # Forward pass with current state on current params
                model_output_death = policy(state_tensor)  # (1, 3, 640, 320)
                
                # Compute loss from LOSS region
                loss_region_death = model_output_death[:, :, 316:324, 156:164]
                loss_death = compute_loss_from_region(loss_region_death.squeeze(0))
                
                # Negate the loss (reverse gradient direction)
                loss2 = -loss_death
                
                # Compute gradients and apply with larger learning rate
                optimizer.zero_grad()
                loss2.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
                
                # Use larger learning rate for death update
                for param_group in optimizer.param_groups:
                    param_group['lr'] = death_lr
                optimizer.step()
                
                # Restore normal learning rate range
                for param_group in optimizer.param_groups:
                    param_group['lr'] = lr_max
                
                print(f"    Applied negative loss gradient with death_lr={death_lr:.0e}, survived {survival_steps} steps")
                
                # Reset environment
                obs, _ = env.reset()
                preprocessor.reset()
                combined_eyes, hp = preprocessor.get_state(obs[0])
                out_next = None  # Reset out_next
                survival_steps = 0  # Reset survival counter
                
                # Don't break, continue training in new episode
                continue
            
            # === Handle done: reset environment and continue training ===
            if done:
                print(f"  [Step {total_steps}] Episode done! Survived {survival_steps} steps. Resetting environment...")
                obs, _ = env.reset()
                preprocessor.reset()
                combined_eyes, hp = preprocessor.get_state(obs[0])
                out_next = None  # Reset out_next
                survival_steps = 0  # Reset survival counter
                continue
        
        # Track episode stats
        scores_deque.append(episode_reward)
        scores.append(episode_reward)
        
        # Save best model
        if episode_reward > best_score:
            best_score = episode_reward
            torch.save(policy.state_dict(), "my_best_godot_policy.pth")

        if i_episode % print_every == 0:
            avg_score = np.mean(scores_deque)
            print(f"Episode {i_episode}\tAverage Score: {avg_score:.2f}\tBest: {best_score:.2f}\tSteps: {total_steps}")

    return scores

In [ ]:
# Hyperparameters for GodotEnv training with MobileNetV3
# Updated: per-step updates with dynamic learning rate and death penalty
hyperparameters = {
    "n_training_episodes": 1,
    "n_evaluation_episodes": 1,
    "max_t": 1000000,  # Increased since we update per-step
    "gamma": 0.99,
    "lr_min": 1e-5,    # Minimum learning rate (very small)
    "lr_max": 2e-4,    # Maximum learning rate (smaller range)
    "death_lr": 1e-3,  # Large learning rate for death penalty update
    "pretrained": True,
    "freeze_backbone": False,
    "n_agents": 2,      # Number of agents in GodotEnv
}

# Create policy and optimizer
godot_policy = MobileNetV3PolicyGodot(
    pretrained=hyperparameters["pretrained"],
    freeze_backbone=hyperparameters["freeze_backbone"],
).to(device)

# Initial optimizer with lr_max (will be dynamically adjusted per step)
godot_optimizer = optim.Adam(
    godot_policy.parameters(), 
    lr=hyperparameters["lr_max"]
)

# Create preprocessor
godot_preprocessor = GodotPreprocessor()

print(f"Training on GodotEnv with MobileNetV3 backbone")
print(f"Device: {device}")
print(f"Pretrained: {hyperparameters['pretrained']}, Freeze backbone: {hyperparameters['freeze_backbone']}")
print(f"Per-step updates with dynamic learning rate: [{hyperparameters['lr_min']:.0e}, {hyperparameters['lr_max']:.0e}]")
print(f"Death penalty: negative loss gradient directly on current params with death_lr={hyperparameters['death_lr']:.0e}")
print(f"Hyperparameters: {hyperparameters}")

# Train the policy
scores = reinforce_godot(
    godot_policy,
    godot_optimizer,
    godot_preprocessor,
    env,
    hyperparameters["n_training_episodes"],
    hyperparameters["max_t"],
    hyperparameters["gamma"],
    print_every=10,
    lr_min=hyperparameters["lr_min"],
    lr_max=hyperparameters["lr_max"],
    death_lr=hyperparameters["death_lr"],
    n_agents=hyperparameters["n_agents"],
)

# Save final model
torch.save(godot_policy.state_dict(), "saved_models/my_godot_mobilenetv3_final.pth")
print("\nTraining complete! Model saved.")

Training on GodotEnv with MobileNetV3 backbone
Device: mps
Pretrained: True, Freeze backbone: False
Per-step updates with dynamic learning rate: [1e-05, 2e-04]
Death penalty: negative loss gradient directly on current params with death_lr=1e-03
Hyperparameters: {'n_training_episodes': 1, 'n_evaluation_episodes': 1, 'max_t': 1000000, 'gamma': 0.99, 'lr_min': 1e-05, 'lr_max': 0.0002, 'death_lr': 0.001, 'pretrained': True, 'freeze_backbone': False, 'n_agents': 1}
  [Step 100] HP=6.0, LR=1.18e-04, Survived=100 steps
  [Step 200] HP=1.0, LR=1.05e-04, Survived=200 steps
  [Step 300] HP=1.0, LR=1.05e-04, Survived=300 steps
  [Step 400] HP=1.0, LR=1.05e-04, Survived=400 steps
  [Step 500] HP=1.0, LR=1.05e-04, Survived=500 steps
  [Step 600] HP=1.0, LR=1.05e-04, Survived=600 steps
  [Step 700] HP=1.0, LR=1.05e-04, Survived=700 steps
  [Step 800] HP=1.0, LR=1.05e-04, Survived=800 steps
  [Step 900] HP=1.0, LR=1.05e-04, Survived=900 steps
  [Step 1000] HP=1.0, LR=1.05e-04, Survived=1000 steps
  [

ConnectionResetError: [Errno 54] Connection reset by peer

: 

In [ ]:
# Close environment when done
env.close()
print("Environment closed.")

close message sent
Environment closed.
